In [1]:
import os

import pandas as pd
import scanpy as sc

import DeconV as dv

In [2]:
wrkdir = os.path.join("..", "..", "..", "..", "data", "pbmc")

In [3]:
adata = sc.read_h5ad(os.path.join(wrkdir, "reference.h5ad"))
bulk = pd.read_table(os.path.join(wrkdir, "bulk.txt"), index_col=0)
bulk.index.name = None

sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
adata = dv.tl.combine(adata, bulk)

adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=None)
adata.layers["ncounts"] = adata.X.copy()
sc.pp.log1p(adata)

scRNA-seq data - cells: 2622, genes: 13293
bulk RNA-seq data - samples: 12, genes: 13293


In [4]:
adata.obs["labels"].unique()

['CD4 T', 'B cells', 'Monocytes', 'NK', 'CD8 T', 'DCs']
Categories (6, object): ['B cells', 'CD4 T', 'CD8 T', 'DCs', 'Monocytes', 'NK']

In [5]:
# Scaden
# bulk
pd.DataFrame(adata.varm["bulk"], index=adata.var_names, columns=adata.uns["bulk_samples"]).to_csv(os.path.join("data", "pbmc_bulk_data.txt"), sep="\t")
# count matrix
pd.DataFrame(adata.layers["ncounts"].A, index=adata.obs_names, columns=adata.var_names).reset_index(drop=True).to_csv(os.path.join("data", "pbmc_counts.txt"), sep="\t")
# cell types
adata.obs.rename(columns={"labels": "Celltype"})["Celltype"].to_csv(os.path.join("data", "pbmc_celltypes.txt"), index=False)